# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing entities by their Croissant `@id` fields for robust and reproducible data analysis.

### Dataset Source
Dataset Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

print("Dataset metadata loaded successfully.")


## 2. Data Overview
Display an overview of the available record sets and fields as defined by their `@id` fields in the Croissant schema.

In [ ]:
# Helper function to print record set info with IDs and field structure

print("Available record sets:\n")
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields:")
        for field in record_set.fields:
            print(f"    - {field.name} (id: {field.id}, dataType: {field.data_type if hasattr(field, 'data_type') else 'unknown'})")
    print()

# Optionally, assign a variable for demonstration
# E.g., record_set_ids = [rs.id for rs in dataset.metadata.record_sets]


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. 
All references to record sets and fields below use their Croissant `@id` values.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# Load all records as DataFrames by record set @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")

# Display DataFrame columns and head for the first record set
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"\nColumns in first record set ({selected_record_set_id}):")
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
This section demonstrates basic filtering, normalization, and grouping, referencing fields only by their `@id`.

You may adjust `numeric_field_id` and `group_field_id` below based on the record set and fields listed above.

In [ ]:
# --- Edit these to match the dataset structure for EDA experiments ---

# Use the first record set as a working example
record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id] if record_set_id else None

# Display available columns for field selection guidance
print(f"Available fields (by @id) in record set {record_set_id}:")
display(df.head())

# Example: Choose a numeric field (edit as appropriate)
# For this dataset, let's guess a relevant field: e.g. 'coverage' or 'MW' (molecular weight) by its @id.
# Find a likely numeric field
numeric_field_candidates = [col for col in df.columns if col.lower() in ['coverage', 'mw', 'molecular_weight', 'abundance', 'mass', 'peptide_count']]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]
else:
    # fallback to the first float-like column
    # try to infer from dtypes or field names
    numeric_field_id = df.select_dtypes(include='number').columns[0] if not df.empty else None

print(f"Selected numeric field for EDA: {numeric_field_id}")

# Filter for values > threshold
threshold = 10
if numeric_field_id and numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (count: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalize this column
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by a categorical field (edit as needed)
    group_field_candidates = [col for col in df.columns if col.lower() in ['description','accession','protein','category','group'] or df[col].dtype=='object']
    group_field_id = None
    for col in group_field_candidates:
        if col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())
else:
    print("No suitable numeric field available for EDA.")

## 5. Visualization
Visualize the distribution of the chosen numeric field and its normalized counterpart, referencing the field via its `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id in filtered_df.columns:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")

    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.subplot(1,2,2)
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True, color='orange')
        plt.title(f"Distribution of Normalized {numeric_field_id}")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field data available for visualization.")

## 6. Conclusion
- Successfully loaded and explored the FAIR² mass spectrometry dataset using the `mlcroissant` library and referenced all data entities by their Croissant `@id`.
- Displayed available record sets, fields, and loaded tabular data from a selected record set using its `@id`.
- Performed simple EDA: filtered, normalized, and grouped by Croissant field `@id`.
- Visualized the distribution of a numeric field. 

This workflow provides a robust, schema-driven, and reproducible way to access scientific datasets for further ML or statistical analysis.